# Breast Cancer Gene Prioritization Pipeline**Disease**: Breast Cancer (BC)This notebook implements a complete network-based gene prioritization pipeline with two parallel approaches:## Pipeline A: STRING-based Discovery (Steps 1–7)1. **Step 1** — Seed gene ingestion from GWAS2. **Step 2** — Full STRING v12.0 PPI network construction3. **Step 3** — Random Walk with Restart (RWR) propagation4. **Step 4** — NetColoc proximity testing (z-scores + BH FDR)5. **Step 5** — Subgraph extraction (significant genes only)6. **Step 6** — Leiden clustering7. **Step 7** — KEGG + Reactome pathway enrichment## Pipeline B: Disease-Specific PPI (Steps 8–14)8. **Step 8** — Disease PPI via PSICQUIC + BioGRID9. **Step 9** — RWR on disease PPI10. **Step 10** — NetColoc proximity testing11. **Step 11** — Subgraph extraction12. **Step 12** — Leiden clustering13. **Step 13** — Module comparison (STRING vs Disease PPI)14. **Step 14** — KEGG + Reactome pathway enrichment

---## SetupInstall required Python and R packages.

In [ ]:
# Install Python dependencies
!pip install -q pandas numpy networkx matplotlib seaborn gseapy leidenalg python-igraph netcoloc statsmodels scipy requests


### Mount Google DriveUpload your `Project` folder to Google Drive under `MyDrive/Project/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/Project'
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Contents: {os.listdir(".")}')


---# Pipeline A: STRING-based Discovery---

## Step 1: Seed Gene IngestionLoad breast cancer seed genes from GWAS catalog reported genes.

In [ ]:
import pandas as pd
import os

os.makedirs("results/genes", exist_ok=True)

raw_df = pd.read_csv("resources/raw/gwas/BC_reported_genes.tsv", sep="\t")

raw_df["source"] = "BC_latest_reported_genes"
raw_df["evidence"] = "reported_gwas_gene"

raw_df.to_csv("results/genes/seeds_BC.tsv", sep="\t", index=False)



## Step 2: Full STRING v12.0 PPI NetworkDownloads the complete human STRING interactome (~12M edges), filters by combined score ≥ 0.7, maps ENSP IDs to gene symbols, and saves the final edge list.**⏱ This step takes ~5-10 minutes** (downloading ~400MB of STRING data).

In [ ]:
# ============================================================================
# step2_string.py
# Full STRING PPI Network Construction
#
# Downloads the complete STRING v12.0 human PPI network, filters by
# combined score >= 0.7, maps ENSP protein IDs to gene symbols, removes
# duplicates and self-loops, and saves the final edge list.
#
# This approach uses the FULL human interactome (not seed-gene-centric API)
# which is the standard approach in network medicine.
#
# Required files (auto-downloaded if not present):
#   resources/raw/string/9606.protein.links.v12.0.txt.gz
#   resources/raw/string/9606.protein.info.v12.0.txt.gz
# ============================================================================

import os
import requests
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR  = "/content/drive/MyDrive/Project"

STRING_DIR   = os.path.join(PROJECT_DIR, "resources", "raw", "string")
LINKS_FILE   = os.path.join(STRING_DIR,  "9606.protein.links.v12.0.txt.gz")
INFO_FILE    = os.path.join(STRING_DIR,  "9606.protein.info.v12.0.txt.gz")

OUTPUT_DIR   = os.path.join(PROJECT_DIR, "results", "networks")
OUTPUT_FILE  = os.path.join(OUTPUT_DIR,  "string_bg.tsv")
PLOT_FILE    = os.path.join(OUTPUT_DIR,  "ppi_network_BC.png")

SCORE_CUTOFF = 0.7   # combined score threshold (STRING scores are 0–1 after /1000)

LINKS_URL = "https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz"
INFO_URL  = "https://stringdb-downloads.org/download/protein.info.v12.0/9606.protein.info.v12.0.txt.gz"

os.makedirs(STRING_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helper: download file if not already present ──────────────────────────────
def download_if_missing(url, dest_path):
    if os.path.exists(dest_path):
        print(f"  Already exists: {dest_path}")
        return
    print(f"  Downloading: {url}")
    print(f"  Saving to  : {dest_path}")
    response = requests.get(url, stream=True)
    response.raise_for_status()
    with open(dest_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    print(f"  Download complete.")

# ── 1. Download STRING files ───────────────────────────────────────────────────
print("Checking STRING raw files...")
download_if_missing(LINKS_URL, LINKS_FILE)
download_if_missing(INFO_URL,  INFO_FILE)

# ── 2. Load full STRING links and filter by score ─────────────────────────────
print("\nLoading STRING protein links...")
df = pd.read_csv(LINKS_FILE, sep=" ", compression="gzip")

# STRING scores are integers 0–1000 → divide by 1000 to get 0–1
df["combined_score"] = df["combined_score"] / 1000.0

print(f"  Total edges (before filter) : {len(df)}")
df = df[df["combined_score"] >= SCORE_CUTOFF]
print(f"  Edges after score >= {SCORE_CUTOFF}    : {len(df)}")

# ── 3. Strip species prefix from protein IDs (9606.ENSP... → ENSP...) ─────────
df["protein1"] = df["protein1"].str.split(".").str[1]
df["protein2"] = df["protein2"].str.split(".").str[1]
df = df.reset_index(drop=True)

# ── 4. Load protein info and build ENSP → gene symbol mapping ─────────────────
print("\nLoading STRING protein info for ENSP → gene symbol mapping...")
info = pd.read_csv(INFO_FILE, sep="\t", compression="gzip")
info["protein_id"] = info["#string_protein_id"].str.split(".").str[1]

# Build mapping dict
protein_to_gene = dict(zip(info["protein_id"], info["preferred_name"]))

# Remove entries where preferred_name is an ENSG/ENSP ID (not a real gene name)
protein_to_gene = {k: v for k, v in protein_to_gene.items() if "ENS" not in v}
print(f"  Proteins with valid gene symbols: {len(protein_to_gene)}")

# ── 5. Map protein IDs to gene symbols ────────────────────────────────────────
print("\nMapping protein IDs to gene symbols...")
edges = df.copy()
edges["geneA"] = edges["protein1"].map(protein_to_gene)
edges["geneB"] = edges["protein2"].map(protein_to_gene)

# Drop edges where either gene could not be mapped
edges = edges.dropna(subset=["geneA", "geneB"])

# Drop self-loops (same gene on both sides)
edges = edges[edges["geneA"] != edges["geneB"]]

print(f"  Edges after gene mapping & self-loop removal: {len(edges)}")

# ── 6. Build final edge list ───────────────────────────────────────────────────
gene_edges = edges[["geneA", "geneB", "combined_score"]].copy()
gene_edges = gene_edges.rename(columns={
    "combined_score": "weight",
    "geneA": "nodeA",
    "geneB": "nodeB"
})

# Remove duplicate edges (A-B and B-A are the same undirected edge)
gene_edges["sorted_pair"] = gene_edges.apply(
    lambda x: tuple(sorted([x["nodeA"], x["nodeB"]])), axis=1
)
gene_edges = gene_edges.drop_duplicates("sorted_pair")
gene_edges = gene_edges.drop(columns="sorted_pair")
gene_edges = gene_edges.reset_index(drop=True)

# Add metadata columns to match your pipeline format
gene_edges["source"]   = "STRING"
gene_edges["evidence"] = "combined_score"
gene_edges["version"]  = "v12.0"

print(f"  Final edges (after dedup)    : {len(gene_edges)}")
print(f"  Unique genes in network      : {len(set(gene_edges['nodeA']) | set(gene_edges['nodeB']))}")

# ── 7. Save edge list ─────────────────────────────────────────────────────────
gene_edges.to_csv(OUTPUT_FILE, sep="\t", index=False)
print(f"\n✅ Saved: {OUTPUT_FILE}")

# ── 8. Quick plot of seed gene subnetwork ─────────────────────────────────────
# (plotting full network is too large — plot seed-gene neighbourhood instead)
print("\nGenerating seed gene network plot...")
seeds_df = pd.read_csv(
    os.path.join(PROJECT_DIR, "results", "genes", "seeds_BC.tsv"), sep="\t"
)
seed_genes = set(seeds_df["gene_symbol"].tolist())

# Filter edges to seed genes only for plotting
seed_edges = gene_edges[
    gene_edges["nodeA"].isin(seed_genes) & gene_edges["nodeB"].isin(seed_genes)
]

G_plot = nx.from_pandas_edgelist(seed_edges, source="nodeA", target="nodeB", edge_attr="weight")

plt.figure(figsize=(12, 12))
nx.draw_networkx(G_plot, node_size=200, with_labels=True, font_size=8, width=1, alpha=0.7)
plt.title("STRING PPI Network — Breast Cancer Seed Genes (score ≥ 0.7)")
plt.savefig(PLOT_FILE, dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Saved: {PLOT_FILE}")

# ── 9. Summary ────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"STRING NETWORK SUMMARY")
print(f"{'='*60}")
print(f"  Source         : STRING v12.0 (full human interactome)")
print(f"  Species        : Homo sapiens (9606)")
print(f"  Score cutoff   : >= {SCORE_CUTOFF}")
print(f"  Total edges    : {len(gene_edges)}")
print(f"  Unique genes   : {len(set(gene_edges['nodeA']) | set(gene_edges['nodeB']))}")
print(f"\n✅ Step 2 complete — saved to:\n   {OUTPUT_FILE}")


Checking STRING raw files...
  Already exists: /content/drive/MyDrive/Project/resources/raw/string/9606.protein.links.v12.0.txt.gz
  Already exists: /content/drive/MyDrive/Project/resources/raw/string/9606.protein.info.v12.0.txt.gz

Loading STRING protein links...
  Total edges (before filter) : 12584304
  Edges after score >= 0.7    : 571026

Loading STRING protein info for ENSP → gene symbol mapping...
  Proteins with valid gene symbols: 19238

Mapping protein IDs to gene symbols...
  Edges after gene mapping & self-loop removal: 478484
  Final edges (after dedup)    : 236333
  Unique genes in network      : 16155

STRING NETWORK SUMMARY
  Source         : STRING v12.0 (full human interactome)
  Species        : Homo sapiens (9606)
  Score cutoff   : >= 0.7
  Total edges    : 236333
  Unique genes   : 16155

✅ Step 2 complete


## Step 3: Random Walk with Restart (RWR)Runs RWR on the STRING network using seed genes to rank all ~16K genes by proximity to seeds.**⏱ This step takes ~2-3 minutes.****⚠️ This step uses R.** Run the file `step3_propagation.R` separately in RStudio or an R environment.**Input:** `results/networks/string_bg.tsv`, `results/genes/seeds_BC.tsv`**Output:** `results/genes/expanded_BC_rwr.tsv`

## Step 4: Network Proximity Testing (NetColoc)Uses degree-binned z-scores with 1000 permutations to identify genes significantly proximal to seed genes in the STRING network. Applies Benjamini-Hochberg FDR correction.**⏱ This step takes ~5-10 minutes.**

In [ ]:
# ============================================================================
# step4_proximity.py
# Network Proximity Testing for RWR Candidate Genes
#
# Uses NetColoc heat propagation z-scores (degree-binned) to test which
# RWR candidate genes are significantly proximal to seed genes in STRING.
#
# Workflow:
#   1. Build normalized adjacency & individual heats matrix (NetColoc)
#   2. Compute degree-binned z-scores (1000 permutations)
#   3. Convert z-scores → raw p-values via scipy.stats.norm.sf(z)
#   4. Apply Benjamini-Hochberg FDR correction
#   5. Filter: adjusted p-value < 0.05
#
# References:
#   [1] Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery
#       rate. JRSS-B, 57(1), 289–300.
#   [2] Seabold, S., & Perktold, J. (2010). Statsmodels. SciPy 2010.
#   [3] Wright, S. et al. (2021). NetColoc. GitHub.
# ============================================================================

import pandas as pd
import numpy as np
import networkx as nx
from scipy import stats
from statsmodels.stats.multitest import multipletests
from netcoloc import netprop, netprop_zscore
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR = "/content/drive/MyDrive/Project"

NETWORK_FILE   = os.path.join(PROJECT_DIR, "results", "networks", "string_bg.tsv")
SEEDS_FILE     = os.path.join(PROJECT_DIR, "results", "genes", "seeds_BC.tsv")
RWR_FILE       = os.path.join(PROJECT_DIR, "results", "genes", "expanded_BC_rwr.tsv")
OUTPUT_FILE    = os.path.join(PROJECT_DIR, "results", "genes", "proximity_BC_rwr.tsv")

FDR_THRESHOLD  = 0.05

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# ── 1. Load STRING PPI and build graph ───────────────────────────────────────
print("=" * 60)
print("STEP 4: Proximity Testing (NetColoc heat propagation)")
print("=" * 60)

edges = pd.read_csv(NETWORK_FILE, sep="\t")
G = nx.from_pandas_edgelist(edges, source="nodeA", target="nodeB")
int_nodes = list(G.nodes())
print(f"\nNetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ── 2. Precompute individual heats matrix (one-time) ─────────────────────────
print("\nCalculating w_prime (normalized adjacency)...")
w_prime = netprop.get_normalized_adjacency_matrix(G, conserve_heat=True)
print("Computing individual heats matrix (one-time, ~2-5 mins)...")
w_double_prime = netprop.get_individual_heats_matrix(w_prime, alpha=0.5)
print("Done.")

# ── 3. Load seed genes ───────────────────────────────────────────────────────
seeds_df   = pd.read_csv(SEEDS_FILE, sep="\t")
seed_genes = list(set(seeds_df["gene_symbol"]) & set(int_nodes))
print(f"\nSeed genes in network: {len(seed_genes)} / {len(seeds_df)}")

# ── 4. Calculate per-gene proximity z-scores (degree-binned, 1000 perms) ─────
print("Calculating proximity z-scores (1000 permutations)...")
z_scores, Fnew, Fnew_rand = netprop_zscore.calculate_heat_zscores(
    w_double_prime,
    int_nodes,
    dict(G.degree()),
    seed_genes,
    num_reps         = 1000,
    minimum_bin_size = 100
)
print("Done.")

nan_z = z_scores.isna().sum()
print(f"NaN z-scores in full network : {nan_z} / {len(z_scores)}")

# ── 5. Load RWR genes and compute raw p-values ───────────────────────────────
rwr_df    = pd.read_csv(RWR_FILE, sep="\t")
rwr_genes = set(rwr_df["gene_symbol"]) & set(int_nodes)

records = []
for gene in rwr_genes:
    if gene in z_scores.index:
        z = z_scores[gene]
        if np.isnan(z):
            continue
        rwr_row = rwr_df.loc[rwr_df["gene_symbol"] == gene]
        records.append({
            "gene_symbol" : gene,
            "z_score"     : z,
            "raw_pvalue"  : stats.norm.sf(z),   # one-tailed: P(Z > z)
            "rwr_score"   : rwr_row["score"].values[0],
            "rwr_rank"    : rwr_row["rank"].values[0]
        })

df = pd.DataFrame(records).sort_values("z_score", ascending=False).reset_index(drop=True)

print(f"\nRWR genes with valid z-scores : {len(df)}")
print(f"NaN p-values                  : {df['raw_pvalue'].isna().sum()}")

# ── 6. Benjamini-Hochberg FDR correction [1][2] ─────────────────────────────
df_valid = df.dropna(subset=["raw_pvalue"]).copy()
df_nan   = df[df["raw_pvalue"].isna()].copy()

print(f"Genes entering BH correction  : {len(df_valid)}")

pvals = np.clip(df_valid["raw_pvalue"].to_numpy(dtype=float), 0, 1)

reject, p_adj, _, _ = multipletests(pvals, alpha=FDR_THRESHOLD, method="fdr_bh")

df_valid["adj_pvalue_bh"] = p_adj
df_valid["significant"]   = reject

if len(df_nan) > 0:
    df_nan["adj_pvalue_bh"] = np.nan
    df_nan["significant"]   = False

df = pd.concat([df_valid, df_nan], ignore_index=True) \
       .sort_values("z_score", ascending=False) \
       .reset_index(drop=True)

# ── 7. Save ──────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_FILE, sep="\t", index=False)

# ── 8. Summary ───────────────────────────────────────────────────────────────
n_sig   = df["significant"].sum()
n_total = len(df)

print(f"\n{'='*60}")
print(f"PROXIMITY TESTING RESULTS")
print(f"{'='*60}")
print(f"  Total candidates tested              : {n_total}")
print(f"  Significant (adj_pvalue_bh < {FDR_THRESHOLD})   : {n_sig}")
print(f"  Not significant                      : {n_total - n_sig}")

print(f"\nTop 20 significant genes (sorted by adj_pvalue_bh):")
top20 = df[df["significant"]].sort_values("adj_pvalue_bh").head(20)
print(top20[[
    "gene_symbol", "z_score", "raw_pvalue", "adj_pvalue_bh", "rwr_score"
]].to_string(index=False))

print(f"\n✅ Step 4 complete — results saved to:\n   {OUTPUT_FILE}")



STEP 4: Proximity Testing (NetColoc heat propagation)

Network: 16155 nodes, 236333 edges

Calculating w_prime (normalized adjacency)...
Computing individual heats matrix (one-time, ~2-5 mins)...
Done.

Seed genes in network: 23 / 25
Calculating proximity z-scores (1000 permutations)...
Done.

RWR genes with valid z-scores : 16155
Genes entering BH correction  : 16155

PROXIMITY TESTING RESULTS
  Total candidates tested              : 16155
  Significant (adj_pvalue_bh < 0.05)   : 253
  Not significant                      : 15902

✅ Step 4 complete


## Step 5: Subgraph ExtractionExtracts the subgraph containing only proximity-significant genes (FDR < 0.05). An edge is kept only if BOTH endpoints are significant.

In [ ]:
# ============================================================================
# step5_subgraph.py
# Subgraph Extraction for Proximity-Significant RWR Genes
#
# Builds a subgraph from the STRING PPI network using only genes that:
#   1. Passed the Benjamini-Hochberg FDR proximity test (adj_pvalue_bh < 0.05)
#   2. Have at least one edge in STRING where BOTH endpoints are in the
#      proximity-significant gene set.
#
# Rule: An edge (A, B) is included only if BOTH gene A AND gene B are present
# in the significant gene set. If either endpoint is absent, the edge is
# dropped. Genes with no surviving edges are excluded from the subgraph.
# ============================================================================

import pandas as pd
import networkx as nx
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR    = "/content/drive/MyDrive/Project"

NETWORK_FILE   = os.path.join(PROJECT_DIR, "results", "networks", "string_bg.tsv")
PROXIMITY_FILE = os.path.join(PROJECT_DIR, "results", "genes",   "proximity_BC_rwr.tsv")
OUTPUT_EDGES   = os.path.join(PROJECT_DIR, "results", "networks", "string_subgraph_BC_rwr.tsv")

# ── 1. Load proximity results and filter significant genes ────────────────────
# Uses the 'significant' column from step4 (adj_pvalue_bh < 0.05).
print("Loading proximity results...")
prox_df = pd.read_csv(PROXIMITY_FILE, sep='\t')

sig_df    = prox_df[prox_df['significant'] == True].copy()
sig_genes = set(sig_df['gene_symbol'].tolist())

print(f"  Total genes tested                    : {len(prox_df)}")
print(f"  Significant (adj_pvalue_bh < 0.05)    : {len(sig_genes)}")

# ── 2. Load STRING PPI network ────────────────────────────────────────────────
print("\nLoading STRING PPI network...")
net_df = pd.read_csv(NETWORK_FILE, sep='\t')
print(f"  Total edges in STRING   : {len(net_df)}")

# ── 3. Filter edges — BOTH endpoints must be in the significant gene set ──────
# Rule: edge (A, B) is kept only if A ∈ sig_genes AND B ∈ sig_genes.
# This ensures the subgraph contains only well-connected significant genes.
# Genes whose ALL neighbours are outside the significant set are automatically
# excluded because none of their edges survive this filter.
print("\nFiltering edges (both endpoints must be in significant gene set)...")
mask      = net_df['nodeA'].isin(sig_genes) & net_df['nodeB'].isin(sig_genes)
sub_edges = net_df[mask].copy()

print(f"  Edges after filtering   : {len(sub_edges)}")

# ── 4. Identify which genes actually appear in the subgraph ───────────────────
nodes_in_subgraph = set(sub_edges['nodeA']).union(set(sub_edges['nodeB']))
excluded = sig_genes - nodes_in_subgraph

print(f"  Genes in subgraph       : {len(nodes_in_subgraph)}")
print(f"  Significant genes with no internal edges (excluded): {len(excluded)}")
if excluded:
    print(f"    Excluded: {', '.join(sorted(excluded))}")

# ── 5. Save subgraph edge list ────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_EDGES), exist_ok=True)
sub_edges.to_csv(OUTPUT_EDGES, sep='\t', index=False)

# ── 6. Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"SUBGRAPH EXTRACTION RESULTS")
print(f"{'='*60}")
print(f"  Input  : {len(sig_genes)} proximity-significant genes (BH FDR < 0.05)")
print(f"  Output : {len(nodes_in_subgraph)} genes, {len(sub_edges)} edges")
print(f"\n✅ Step 5 subgraph complete — saved to:\n   {OUTPUT_EDGES}")


Loading proximity results...
  Total genes tested                    : 16155
  Significant (adj_pvalue_bh < 0.05)    : 253

Loading STRING PPI network...
  Total edges in STRING   : 236333

Filtering edges (both endpoints must be in significant gene set)...
  Edges after filtering   : 2389
  Genes in subgraph       : 252
  Significant genes with no internal edges (excluded): 1

SUBGRAPH EXTRACTION RESULTS
  Input  : 253 proximity-significant genes (BH FDR < 0.05)
  Output : 252 genes, 2389 edges

✅ Step 5 subgraph complete


## Step 6: Leiden Clustering (STRING)Runs Leiden clustering on the proximity-filtered subgraph. Uses LCC only and removes clusters with fewer than 20 genes.

In [ ]:
# ============================================================================
# step6_clustering.py
# Leiden Clustering on the Proximity-Filtered STRING Subgraph
#
# Loads the subgraph built in step5_subgraph.py and runs Leiden clustering.
# Clusters with fewer than 20 genes are dropped as too small to be meaningful.
# ============================================================================

import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR  = "/content/drive/MyDrive/Project"

SUBGRAPH_FILE  = os.path.join(PROJECT_DIR, "results", "networks", "string_subgraph_BC_rwr.tsv")
PROXIMITY_FILE = os.path.join(PROJECT_DIR, "results", "genes",   "proximity_BC_rwr.tsv")
OUTPUT_FILE    = os.path.join(PROJECT_DIR, "results", "modules", "modules_BC_leiden.tsv")

MIN_CLUSTER_SIZE = 20   # drop clusters smaller than this

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
edges   = pd.read_csv(SUBGRAPH_FILE,  sep='\t')
rwr_top = pd.read_csv(PROXIMITY_FILE, sep='\t')

# RWR score lookup {gene -> rwr_score}
rwr_lookup = dict(zip(rwr_top["gene_symbol"], rwr_top["rwr_score"]))

# Only use proximity-significant genes
sig_mask = rwr_top["significant"] == True
selected_nodes = set(rwr_top.loc[sig_mask, "gene_symbol"])

# ── Build subgraph with RWR-based edge weights ────────────────────────────────
G = nx.Graph()

for _, row in edges.iterrows():
    a, b = row["nodeA"], row["nodeB"]
    if a in selected_nodes and b in selected_nodes:
        rwr_a = rwr_lookup.get(a, 0.0)
        rwr_b = rwr_lookup.get(b, 0.0)
        rwr_edge_weight = (rwr_a + rwr_b) / 2.0   # mean RWR score as edge weight
        G.add_edge(a, b, weight=rwr_edge_weight)

print(f"Subgraph nodes : {G.number_of_nodes()}")
print(f"Subgraph edges : {G.number_of_edges()}")

# ── Keep largest connected component ─────────────────────────────────────────
lcc_nodes = max(nx.connected_components(G), key=len)
G = G.subgraph(lcc_nodes).copy()

print(f"LCC nodes : {G.number_of_nodes()}")
print(f"LCC edges : {G.number_of_edges()}")

# ── Convert to igraph ─────────────────────────────────────────────────────────
node_list       = list(G.nodes())
mapping         = {node: i for i, node in enumerate(node_list)}
reverse_mapping = {i: node for node, i in mapping.items()}

edges_with_weight = [
    (mapping[u], mapping[v], data["weight"])
    for u, v, data in G.edges(data=True)
]

ig_graph = ig.Graph(
    n        = len(node_list),
    edges    = [(u, v) for u, v, _ in edges_with_weight],
    directed = False
)
ig_graph.es["weight"] = [w for _, _, w in edges_with_weight]
ig_graph.vs["name"]   = node_list

# ── Leiden clustering ─────────────────────────────────────────────────────────
leiden_partition = leidenalg.find_partition(
    ig_graph,
    leidenalg.RBConfigurationVertexPartition,
    weights          = "weight",
    resolution_parameter = 1.0,
    seed             = 42,
    n_iterations     = 50
)

print(f"\nClusters found : {len(leiden_partition)}")
print(f"Modularity     : {leiden_partition.modularity:.4f}")

# ── Extract results ───────────────────────────────────────────────────────────
leiden_clusters = {
    reverse_mapping[node_id]: comm
    for node_id, comm in enumerate(leiden_partition.membership)
}

cluster_df = pd.DataFrame([
    {
        "gene"     : gene,
        "cluster"  : cluster,
        "rwr_score": rwr_lookup.get(gene, 0.0),
    }
    for gene, cluster in leiden_clusters.items()
])

# ── Drop clusters with fewer than 20 genes ────────────────────────────────────
# FIX 1: threshold raised from 5 → 20 as instructed
valid_clusters = cluster_df["cluster"].value_counts()
valid_clusters = valid_clusters[valid_clusters >= MIN_CLUSTER_SIZE].index
cluster_df     = cluster_df[cluster_df["cluster"].isin(valid_clusters)]

cluster_df = cluster_df.sort_values(["cluster", "rwr_score"], ascending=[True, False])

print(f"\nCluster summary (clusters with >= {MIN_CLUSTER_SIZE} genes):")
print(cluster_df.groupby("cluster").agg(
    n_genes  = ("gene",      "count"),
    mean_rwr = ("rwr_score", "mean"),
    max_rwr  = ("rwr_score", "max")
).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
# FIX 2: save cluster_df (filtered) instead of df (all nodes in G)
cluster_df.to_csv(OUTPUT_FILE, sep='\t', index=False)

print(f"\nClustering complete.")
print(f"Leiden clusters (>= {MIN_CLUSTER_SIZE} genes): {cluster_df['cluster'].nunique()}")
print(f"Total genes saved : {len(cluster_df)}")
print(f"\n✅ Step 6 complete — saved to:\n   {OUTPUT_FILE}")


Subgraph nodes : 252
Subgraph edges : 2389
LCC nodes : 251
LCC edges : 2389

Clusters found : 6
Modularity     : 0.3412

Cluster summary (clusters with >= 20 genes):
         n_genes  mean_rwr   max_rwr
cluster
0             94  0.000312  0.001303
1             84  0.000432  0.001917
2             34  0.000208  0.000592
3             25  0.000188  0.000331

Clustering complete.
Leiden clusters (>= 20 genes): 4
Total genes saved : 237

✅ Step 6 complete


## Step 7: Pathway Enrichment (STRING Modules)Runs KEGG and Reactome enrichment for each STRING module using Enrichr.Only terms with Adjusted P-value < 0.05 are kept.

In [ ]:
# ============================================================================
# step7_enrichment.py
# Pathway Enrichment Analysis per Leiden Module
#
# For each module identified in step6_clustering.py, runs Enrichr enrichment
# against KEGG and Reactome databases and saves results and plots per module.
#
# Notes:
#   1. Skips plotting/saving when a cluster has no enrichment results
#   2. Uses only Adjusted P-value < 0.05 for significance (removes
#      non-significant terms before saving/plotting)
#   3. Saves to per-module folders (no overwriting)
#
# Input : results/modules/modules_BC_leiden.tsv
# Output: results/enrichment/module_*/  (one folder per module)
#         results/enrichment/all_modules_enrichment.csv  (combined)
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import gseapy as gp
except ImportError:
    raise ImportError("gseapy not installed. Run: pip install gseapy")

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR    = "/content/drive/MyDrive/Project"

MODULES_FILE   = os.path.join(PROJECT_DIR, "results", "modules", "modules_BC_leiden.tsv")
ENRICHMENT_DIR = os.path.join(PROJECT_DIR, "results", "enrichment")

GENE_SETS      = ["KEGG_2021_Human", "Reactome_2022"]
FDR_CUTOFF     = 0.05
MIN_GENES      = 10
TOP_N          = 10

os.makedirs(ENRICHMENT_DIR, exist_ok=True)

# ── 1. Load clustering results ────────────────────────────────────────────────
print("Loading module assignments...")
modules_df = pd.read_csv(MODULES_FILE, sep='\t')
print(f"  Total genes loaded : {len(modules_df)}")
print(f"  Modules found      : {modules_df['cluster'].nunique()}")
print(f"  Columns            : {list(modules_df.columns)}")

# ── 2. Run enrichment per module ──────────────────────────────────────────────
all_results = []

for cluster_id in sorted(modules_df['cluster'].unique()):

    module_genes = (
        modules_df[modules_df['cluster'] == cluster_id]['gene']
        .dropna()
        .astype(str)
        .tolist()
    )

    print(f"\n{'='*60}")
    print(f"Module {cluster_id} — {len(module_genes)} genes")
    print(f"{'='*60}")

    if len(module_genes) < MIN_GENES:
        print(f"  Skipping — fewer than {MIN_GENES} genes")
        continue

    # Output folder for this module
    module_dir = os.path.join(ENRICHMENT_DIR, f"module_{cluster_id}")
    os.makedirs(module_dir, exist_ok=True)

    # ── Run Enrichr ───────────────────────────────────────────────────────────
    try:
        enr = gp.enrichr(
            gene_list   = module_genes,
            gene_sets   = GENE_SETS,
            organism    = "human",
            outdir      = os.path.join(module_dir, "enrichr_raw"),
            cutoff      = 1.0,        # get ALL terms, filter below
            no_plot     = True,
        )
        enrich_df = enr.results.copy()
    except Exception as e:
        print(f"  Enrichr failed for module {cluster_id}: {e}")
        continue

    # 1. Skip empty results
    if enrich_df is None or enrich_df.empty:
        print(f"  No enrichment results → skip")
        continue

    # 2. Use Adjusted P-value only — remove non-significant terms
    enrich_df = enrich_df[enrich_df["Adjusted P-value"] < FDR_CUTOFF].copy()

    if enrich_df.empty:
        print(f"  No significant pathways (Adjusted P-value < {FDR_CUTOFF}) → skip")
        continue

    # Add module info
    enrich_df['module'] = cluster_id
    enrich_df['n_genes_in_module'] = len(module_genes)

    # 3. Save per-module (no overwriting)
    enrich_df.to_csv(os.path.join(module_dir, "enrichment_full.csv"), index=False)

    # ── Extract top pathways ──────────────────────────────────────────────────
    top_kegg = (
        enrich_df[enrich_df['Gene_set'] == 'KEGG_2021_Human']
        .sort_values('Adjusted P-value')
        .head(TOP_N)
        .copy()
    )

    top_reactome = (
        enrich_df[enrich_df['Gene_set'] == 'Reactome_2022']
        .sort_values('Adjusted P-value')
        .head(TOP_N)
        .copy()
    )

    # Print summary
    print(f"\n  Top KEGG pathways:")
    if not top_kegg.empty:
        for _, row in top_kegg.head(5).iterrows():
            print(f"    {row['Term'][:60]:<60}  adj_p={row['Adjusted P-value']:.2e}")
    else:
        print("    None found")

    print(f"\n  Top Reactome pathways:")
    if not top_reactome.empty:
        for _, row in top_reactome.head(5).iterrows():
            print(f"    {row['Term'][:60]:<60}  adj_p={row['Adjusted P-value']:.2e}")
    else:
        print("    None found")

    # ── Save CSVs ─────────────────────────────────────────────────────────────
    top_kegg.to_csv(os.path.join(module_dir, "top_kegg.csv"), index=False)
    top_reactome.to_csv(os.path.join(module_dir, "top_reactome.csv"), index=False)

    # ── Plot KEGG (only if significant terms exist) ──────────────────────────
    if not top_kegg.empty:
        top_kegg['-log10(padj)'] = -np.log10(
            top_kegg['Adjusted P-value'].replace(0, 1e-300)
        )
        top_kegg['Term'] = top_kegg['Term'].apply(
            lambda x: x if len(x) <= 55 else x[:52] + '...'
        )
        plt.figure(figsize=(10, 6))
        sns.barplot(data=top_kegg, y='Term', x='-log10(padj)', color='skyblue')
        plt.title(f"Module {cluster_id} — Top {TOP_N} KEGG Pathways ({len(module_genes)} genes)")
        plt.xlabel('-log10(Adjusted P-value)')
        plt.ylabel('Pathway')
        plt.tight_layout()
        plt.savefig(os.path.join(module_dir, "top_kegg.png"), dpi=150)
        plt.close()

    # ── Plot Reactome (only if significant terms exist) ──────────────────────
    if not top_reactome.empty:
        top_reactome['-log10(padj)'] = -np.log10(
            top_reactome['Adjusted P-value'].replace(0, 1e-300)
        )
        top_reactome['Term'] = top_reactome['Term'].apply(
            lambda x: x if len(x) <= 55 else x[:52] + '...'
        )
        plt.figure(figsize=(10, 6))
        sns.barplot(data=top_reactome, y='Term', x='-log10(padj)', color='lightgreen')
        plt.title(f"Module {cluster_id} — Top {TOP_N} Reactome Pathways ({len(module_genes)} genes)")
        plt.xlabel('-log10(Adjusted P-value)')
        plt.ylabel('Pathway')
        plt.tight_layout()
        plt.savefig(os.path.join(module_dir, "top_reactome.png"), dpi=150)
        plt.close()

    # Collect for combined output
    all_results.append(enrich_df)
    print(f"\n  ✅ Module {cluster_id} saved to: {module_dir}")

# ── 3. Save combined results across all modules ──────────────────────────────
if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    combined_df = combined_df.sort_values(['module', 'Adjusted P-value'])
    combined_path = os.path.join(ENRICHMENT_DIR, "all_modules_enrichment.csv")
    combined_df.to_csv(combined_path, index=False)
    print(f"\n{'='*60}")
    print(f"ENRICHMENT COMPLETE")
    print(f"{'='*60}")
    print(f"  Modules processed    : {modules_df['cluster'].nunique()}")
    print(f"  Total significant    : {len(combined_df)}")
    print(f"  Combined results     : {combined_path}")
    print(f"\n  Per-module folders:")
    for cluster_id in sorted(modules_df['cluster'].unique()):
        print(f"    results/enrichment/module_{cluster_id}/")
    print(f"\n✅ Step 7 complete")
else:
    print("\n⚠️  No enrichment results generated.")


---# Pipeline B: Disease-Specific PPI---

## Step 8: Disease PPI via PSICQUIC + BioGRIDBuilds a disease-specific PPI by querying IntAct via PSICQUIC using disease keywords ("breast cancer", "breast neoplasm", "breast carcinoma"). Integrates with BioGRID physical interactions filtered to disease-relevant genes.**⏱ This step takes ~5-10 minutes** (PSICQUIC API queries).

In [ ]:
# ============================================================================
# step8_disease_ppi.py
# Disease-Specific PPI Network via PSICQUIC + BioGRID
#
# Builds a disease-specific PPI by querying IntAct via PSICQUIC using
# the disease term "breast cancer" (NOT by filtering around seed genes).
# Also integrates BioGRID physical interactions for genes found in the
# IntAct disease query.
#
# This produces a disease-contextualized network independent of seed genes,
# allowing a fair comparison with the STRING-based pipeline.
#
# References:
#   [1] del-Toro, N. et al. (2013). A new reference implementation of the
#       PSICQUIC web service. Nucleic Acids Research, 41(W1), W601–W606.
#   [2] Oughtred, R. et al. (2021). The BioGRID database. Nucleic Acids Res.
# ============================================================================

import pandas as pd
import numpy as np
import requests
import zipfile
import subprocess
import re
import os
import time
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR    = "/content/drive/MyDrive/Project"
DISEASE_ABBR   = "BC"
DISEASE_TERMS  = ["breast cancer", "breast neoplasm", "breast carcinoma"]

SEEDS_FILE     = os.path.join(PROJECT_DIR, "results", "genes", "seeds_BC.tsv")
BIOGRID_ZIP    = os.path.join(PROJECT_DIR, "resources", "biogrid",
                               "BIOGRID-MV-Physical-LATEST.tab3.zip")
OUT_FILE       = os.path.join(PROJECT_DIR, "results", "networks",
                               f"ppi_disease_{DISEASE_ABBR}.tsv")
BIOGRID_URL    = ("https://downloads.thebiogrid.org/Download/BioGRID/"
                  "Latest-Release/BIOGRID-MV-Physical-LATEST.tab3.zip")

MI_SCORE_CUTOFF = 0.4   # IntAct MI score threshold
PSICQUIC_BATCH  = 2000  # records per PSICQUIC page
MAX_RECORDS     = 100000  # cap to keep download time reasonable

os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)

print("=" * 60)
print(f"STEP 8: Disease-Specific PPI Network (PSICQUIC + BioGRID)")
print("=" * 60)

# ── Load seeds (for reporting only — NOT used for filtering) ──────────────────
seeds_df   = pd.read_csv(SEEDS_FILE, sep="\t")
seed_set   = set(seeds_df["gene_symbol"].str.strip().str.upper())
print(f"\nSeed genes (for reference): {len(seed_set)}")

# ══════════════════════════════════════════════════════════════════════════════
# SOURCE 1: IntAct via PSICQUIC [1]
# Query by disease term, NOT by seed genes
# ══════════════════════════════════════════════════════════════════════════════
PSICQUIC_BASE = ("https://www.ebi.ac.uk/Tools/webservices/psicquic/"
                 "intact/webservices/current/search/query/")

GENE_PAT = re.compile(r'(?:uniprotkb|hgnc):([A-Z][A-Z0-9\-]+)\(gene name\)', re.I)

def query_psicquic(disease_term, max_records=MAX_RECORDS):
    """Query IntAct PSICQUIC for human-human interactions with disease term."""
    query = f'species:9606 AND species:9606 AND disease:"{disease_term}"'
    encoded = requests.utils.quote(query)

    print(f"\n  Querying PSICQUIC: {disease_term}")

    # Get count first
    count_url = f"{PSICQUIC_BASE}{encoded}?format=count"
    try:
        resp = requests.get(count_url, timeout=30)
        total = int(resp.text.strip())
        print(f"  Total results: {total}")
    except Exception as e:
        print(f"  ❌ Count failed: {e}")
        return []

    # Download in batches
    records = []
    n_download = min(total, max_records)
    print(f"  Downloading up to {n_download} records...")

    for start in range(0, n_download, PSICQUIC_BATCH):
        url = (f"{PSICQUIC_BASE}{encoded}"
               f"?format=tab25&firstResult={start}&maxResults={PSICQUIC_BATCH}")
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code != 200:
                print(f"    ⚠ HTTP {resp.status_code} at offset {start}")
                continue

            lines = resp.text.strip().split("\n")
            batch_kept = 0

            for line in lines:
                if not line.strip():
                    continue
                fields = line.split("\t")
                if len(fields) < 15:
                    continue

                # Parse taxids (columns 9,10 in MITAB 2.5)
                tax_a, tax_b = fields[9], fields[10]
                if "9606" not in tax_a or "9606" not in tax_b:
                    continue   # skip non-human

                # Parse gene names from aliases (columns 4,5)
                alias_a, alias_b = fields[4], fields[5]
                genes_a = GENE_PAT.findall(alias_a)
                genes_b = GENE_PAT.findall(alias_b)

                if not genes_a or not genes_b:
                    continue

                gene_a = genes_a[0].upper()
                gene_b = genes_b[0].upper()

                if gene_a == gene_b:
                    continue   # skip self-loops

                # Parse MI score (column 14)
                conf = fields[14] if len(fields) > 14 else ""
                mi_match = re.search(r'intact-miscore:([\d.]+)', conf)
                weight = float(mi_match.group(1)) if mi_match else 0.0

                if weight < MI_SCORE_CUTOFF:
                    continue

                records.append({
                    "geneA": gene_a,
                    "geneB": gene_b,
                    "weight": weight
                })
                batch_kept += 1

            if (start // PSICQUIC_BATCH) % 10 == 0:
                print(f"    Batch {start}-{start+PSICQUIC_BATCH}: "
                      f"{len(lines)} raw → {batch_kept} kept")

            time.sleep(0.2)   # be nice to the API

        except Exception as e:
            print(f"    ⚠ Error at offset {start}: {e}")
            continue

    print(f"  ✅ IntAct PSICQUIC: {len(records)} interactions kept")
    return records


# Query with multiple disease terms and combine
all_intact = []
for term in DISEASE_TERMS:
    results = query_psicquic(term)
    all_intact.extend(results)

if all_intact:
    df_intact = pd.DataFrame(all_intact)
    df_intact["source"] = "IntAct"
    # Deduplicate within IntAct (same edge from different terms)
    df_intact["pair"] = df_intact.apply(
        lambda r: tuple(sorted([r["geneA"], r["geneB"]])), axis=1)
    df_intact = (df_intact.groupby("pair")
                 .agg(geneA=("geneA", "first"),
                      geneB=("geneB", "first"),
                      weight=("weight", "max"),
                      source=("source", "first"))
                 .reset_index(drop=True))
    print(f"\n  IntAct deduplicated: {len(df_intact)} unique edges")
else:
    df_intact = pd.DataFrame(columns=["geneA", "geneB", "weight", "source"])
    print("\n  ⚠ No IntAct results")

# ══════════════════════════════════════════════════════════════════════════════
# SOURCE 2: BioGRID [2]
# Use ALL human physical interactions (BioGRID doesn't support disease queries)
# ══════════════════════════════════════════════════════════════════════════════
def download_biogrid():
    """Download BioGRID if not present."""
    if os.path.exists(BIOGRID_ZIP):
        try:
            zipfile.ZipFile(BIOGRID_ZIP).testzip()
            print(f"  ✅ BioGRID exists: {os.path.basename(BIOGRID_ZIP)}")
            return
        except:
            os.remove(BIOGRID_ZIP)

    print(f"  ⬇  Downloading BioGRID...")
    os.makedirs(os.path.dirname(BIOGRID_ZIP), exist_ok=True)
    subprocess.run(["curl", "-L", "-o", BIOGRID_ZIP, BIOGRID_URL],
                   check=True, capture_output=True)
    print(f"  ✅ Downloaded ({os.path.getsize(BIOGRID_ZIP)/1e6:.1f} MB)")


def parse_biogrid(zip_path, disease_genes):
    """Parse BioGRID for human interactions involving disease-relevant genes."""
    print(f"\nParsing BioGRID (filtered by {len(disease_genes)} disease genes)...")
    results = []

    with zipfile.ZipFile(zip_path) as z:
        tab3_file = next(
            (n for n in z.namelist() if n.endswith(".tab3.txt")), None)
        if not tab3_file:
            print("  ❌ No TAB3 file found")
            return pd.DataFrame(columns=["geneA", "geneB", "weight"])

        with z.open(tab3_file) as f:
            chunks = pd.read_csv(f, sep="\t", chunksize=50000,
                                 low_memory=False, header=0)

            for chunk_num, chunk in enumerate(chunks):
                chunk.columns = [c.strip().lstrip("#").strip()
                                 for c in chunk.columns]

                sym_a = next((c for c in chunk.columns
                              if "official symbol" in c.lower()
                              and "interactor a" in c.lower()), None)
                sym_b = next((c for c in chunk.columns
                              if "official symbol" in c.lower()
                              and "interactor b" in c.lower()), None)
                org_a = next((c for c in chunk.columns
                              if "organism" in c.lower()
                              and "interactor a" in c.lower()), None)
                org_b = next((c for c in chunk.columns
                              if "organism" in c.lower()
                              and "interactor b" in c.lower()), None)

                if not sym_a or not sym_b:
                    continue

                # Human filter
                if org_a and org_b:
                    human = (
                        chunk[org_a].astype(str).str.contains(
                            "9606|Homo sapiens", na=False, case=False) &
                        chunk[org_b].astype(str).str.contains(
                            "9606|Homo sapiens", na=False, case=False)
                    )
                    chunk = chunk[human].copy()

                if chunk.empty:
                    continue

                chunk["geneA"] = chunk[sym_a].astype(str).str.strip().str.upper()
                chunk["geneB"] = chunk[sym_b].astype(str).str.strip().str.upper()
                chunk["weight"] = 1.0   # BioGRID: uniform weight

                chunk = chunk[~chunk["geneA"].isin(["-", "nan", ""])]
                chunk = chunk[~chunk["geneB"].isin(["-", "nan", ""])]
                chunk = chunk[chunk["geneA"] != chunk["geneB"]]

                # Disease gene filter — BOTH genes must be in disease set
                disease_mask = (chunk["geneA"].isin(disease_genes) &
                                chunk["geneB"].isin(disease_genes))
                chunk = chunk[disease_mask].copy()

                if chunk.empty:
                    continue

                results.append(chunk[["geneA", "geneB", "weight"]])

    if not results:
        return pd.DataFrame(columns=["geneA", "geneB", "weight"])

    df = pd.concat(results, ignore_index=True)
    df["source"] = "BioGRID"
    print(f"  ✅ BioGRID disease-filtered edges: {len(df)}")
    return df


# Extract disease gene set from IntAct PSICQUIC results
intact_genes = set()
if len(df_intact) > 0:
    intact_genes = set(df_intact["geneA"]) | set(df_intact["geneB"])
print(f"\n  Disease gene set from IntAct: {len(intact_genes)} genes")

print(f"\n{'='*50}")
print("Checking BioGRID data...")
download_biogrid()
df_biogrid = parse_biogrid(BIOGRID_ZIP, intact_genes)

# ══════════════════════════════════════════════════════════════════════════════
# MERGE IntAct + BioGRID
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'='*50}")
print(f"IntAct edges  : {len(df_intact)}")
print(f"BioGRID edges : {len(df_biogrid)}")

df_all = pd.concat([df_intact, df_biogrid], ignore_index=True)

# Normalize to undirected pairs
df_all["pair"] = df_all.apply(
    lambda r: tuple(sorted([r["geneA"], r["geneB"]])), axis=1)

# For duplicates — keep max weight
df_merged = (df_all.groupby("pair")
             .agg(weight=("weight", "max"),
                  sources=("source", lambda x: "+".join(sorted(set(x)))))
             .reset_index()
             .assign(
                 geneA=lambda x: x["pair"].apply(lambda p: p[0]),
                 geneB=lambda x: x["pair"].apply(lambda p: p[1])
             )[["geneA", "geneB", "weight", "sources"]])

# Rename columns for pipeline compatibility
df_merged = df_merged.rename(columns={"geneA": "nodeA", "geneB": "nodeB"})

# Summary
all_genes     = set(df_merged["nodeA"]) | set(df_merged["nodeB"])
seeds_covered = seed_set & all_genes
both_sources  = df_merged[df_merged["sources"].str.contains(r"\+")]

print(f"\n✅ Total merged edges     : {len(df_merged)}")
print(f"✅ Unique genes           : {len(all_genes)}")
print(f"✅ Seeds in network       : {len(seeds_covered)}/{len(seed_set)}")
print(f"✅ Edges in BOTH sources  : {len(both_sources)}  ← high confidence")

# Save
Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
df_merged.to_csv(OUT_FILE, sep="\t", index=False)
print(f"\n✅ Saved → {OUT_FILE}")




STEP 8: Disease-Specific PPI Network (PSICQUIC + BioGRID)

Seed genes (for reference): 25

  Querying PSICQUIC: breast cancer
  ✅ IntAct PSICQUIC: 39341 interactions kept

  Querying PSICQUIC: breast neoplasm
  ✅ IntAct PSICQUIC: 39341 interactions kept

  Querying PSICQUIC: breast carcinoma
  ✅ IntAct PSICQUIC: 39341 interactions kept

  IntAct deduplicated: 30432 unique edges
  Disease gene set from IntAct: 9504 genes

Parsing BioGRID (filtered by 9504 disease genes)...
  ✅ BioGRID disease-filtered edges: 237465

IntAct edges  : 30432
BioGRID edges : 237465

✅ Total merged edges     : 87645
✅ Unique genes           : 9504
✅ Seeds in network       : 22/25
✅ Edges in BOTH sources  : 8387  ← high confidence


## Step 9: RWR on Disease PPIRuns Random Walk with Restart on the disease PPI network using the same seed genes.**⚠️ This step uses R.** Run the file `step9_disease_propagation.R` separately in RStudio or an R environment.**Input:** `results/networks/ppi_disease_BC.tsv`, `results/genes/seeds_BC.tsv`**Output:** `results/genes/expanded_BC_disease_rwr.tsv`

## Step 10: Proximity Testing on Disease PPINetColoc proximity testing on the disease PPI network.**⏱ This step takes ~5-10 minutes.**

In [ ]:
# ============================================================================
# step10_disease_proximity.py
# Network Proximity Testing on Disease PPI (NetColoc)
# Same methodology as step4 but on the disease PPI network.
#
# References:
#   [1] Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery
#       rate. JRSS-B, 57(1), 289–300.
#   [2] Seabold, S., & Perktold, J. (2010). Statsmodels. SciPy 2010.
#   [3] Wright, S. et al. (2021). NetColoc. GitHub.
# ============================================================================

import pandas as pd
import numpy as np
import networkx as nx
from scipy import stats
from statsmodels.stats.multitest import multipletests
from netcoloc import netprop, netprop_zscore
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR = "/content/drive/MyDrive/Project"

NETWORK_FILE   = os.path.join(PROJECT_DIR, "results", "networks", "ppi_disease_BC.tsv")
SEEDS_FILE     = os.path.join(PROJECT_DIR, "results", "genes", "seeds_BC.tsv")
RWR_FILE       = os.path.join(PROJECT_DIR, "results", "genes", "expanded_BC_disease_rwr.tsv")
OUTPUT_FILE    = os.path.join(PROJECT_DIR, "results", "genes", "proximity_BC_disease_rwr.tsv")

FDR_THRESHOLD  = 0.05

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# ── 1. Load Disease PPI and build graph ──────────────────────────────────────
print("=" * 60)
print("STEP 10: Proximity Testing on Disease PPI (NetColoc)")
print("=" * 60)

edges = pd.read_csv(NETWORK_FILE, sep="\t")
G = nx.from_pandas_edgelist(edges, source="nodeA", target="nodeB")
int_nodes = list(G.nodes())
print(f"\nNetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ── 2. Precompute individual heats matrix ────────────────────────────────────
print("\nCalculating w_prime (normalized adjacency)...")
w_prime = netprop.get_normalized_adjacency_matrix(G, conserve_heat=True)
print("Computing individual heats matrix (one-time, ~2-5 mins)...")
w_double_prime = netprop.get_individual_heats_matrix(w_prime, alpha=0.5)
print("Done.")

# ── 3. Load seed genes ──────────────────────────────────────────────────────
seeds_df   = pd.read_csv(SEEDS_FILE, sep="\t")
seed_genes = list(set(seeds_df["gene_symbol"]) & set(int_nodes))
print(f"\nSeed genes in network: {len(seed_genes)} / {len(seeds_df)}")

# ── 4. Calculate per-gene proximity z-scores (degree-binned, 1000 perms) ────
print("Calculating proximity z-scores (1000 permutations)...")
z_scores, Fnew, Fnew_rand = netprop_zscore.calculate_heat_zscores(
    w_double_prime,
    int_nodes,
    dict(G.degree()),
    seed_genes,
    num_reps         = 1000,
    minimum_bin_size = 100
)
print("Done.")

nan_z = z_scores.isna().sum()
print(f"NaN z-scores in full network : {nan_z} / {len(z_scores)}")

# ── 5. Load RWR genes and compute raw p-values ──────────────────────────────
rwr_df    = pd.read_csv(RWR_FILE, sep="\t")
rwr_genes = set(rwr_df["gene_symbol"]) & set(int_nodes)

records = []
for gene in rwr_genes:
    if gene in z_scores.index:
        z = z_scores[gene]
        if np.isnan(z):
            continue
        rwr_row = rwr_df.loc[rwr_df["gene_symbol"] == gene]
        records.append({
            "gene_symbol" : gene,
            "z_score"     : z,
            "raw_pvalue"  : stats.norm.sf(z),
            "rwr_score"   : rwr_row["score"].values[0],
            "rwr_rank"    : rwr_row["rank"].values[0]
        })

df = pd.DataFrame(records).sort_values("z_score", ascending=False).reset_index(drop=True)

print(f"\nRWR genes with valid z-scores : {len(df)}")
print(f"NaN p-values                  : {df['raw_pvalue'].isna().sum()}")

# ── 6. Benjamini-Hochberg FDR correction ─────────────────────────────────────
df_valid = df.dropna(subset=["raw_pvalue"]).copy()
df_nan   = df[df["raw_pvalue"].isna()].copy()

print(f"Genes entering BH correction  : {len(df_valid)}")

pvals = np.clip(df_valid["raw_pvalue"].to_numpy(dtype=float), 0, 1)

reject, p_adj, _, _ = multipletests(pvals, alpha=FDR_THRESHOLD, method="fdr_bh")

df_valid["adj_pvalue_bh"] = p_adj
df_valid["significant"]   = reject

if len(df_nan) > 0:
    df_nan["adj_pvalue_bh"] = np.nan
    df_nan["significant"]   = False

df = pd.concat([df_valid, df_nan], ignore_index=True) \
       .sort_values("z_score", ascending=False) \
       .reset_index(drop=True)

# ── 7. Save ──────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_FILE, sep="\t", index=False)

# ── 8. Summary ───────────────────────────────────────────────────────────────
n_sig   = df["significant"].sum()
n_total = len(df)

print(f"\n{'='*60}")
print(f"PROXIMITY TESTING RESULTS (Disease PPI)")
print(f"{'='*60}")
print(f"  Total candidates tested              : {n_total}")
print(f"  Significant (adj_pvalue_bh < {FDR_THRESHOLD})   : {n_sig}")
print(f"  Not significant                      : {n_total - n_sig}")

print(f"\nTop 20 significant genes (sorted by adj_pvalue_bh):")
top20 = df[df["significant"]].sort_values("adj_pvalue_bh").head(20)
print(top20[[
    "gene_symbol", "z_score", "raw_pvalue", "adj_pvalue_bh", "rwr_score"
]].to_string(index=False))

print(f"\n✅ Step 10 complete — results saved to:\n   {OUTPUT_FILE}")



STEP 10: Proximity Testing on Disease PPI (NetColoc)

Network: 9504 nodes, 87645 edges

Seed genes in network: 22 / 25
Calculating proximity z-scores (1000 permutations)...
Done.

PROXIMITY TESTING RESULTS (Disease PPI)
  Total candidates tested              : 9450
  Significant (adj_pvalue_bh < 0.05)   : 203
  Not significant                      : 9247

✅ Step 10 complete


## Step 11: Disease Subgraph ExtractionExtracts subgraph of significant genes from the disease PPI.

In [ ]:
# ============================================================================
# step11_disease_subgraph.py
# Subgraph Extraction for Disease PPI
# Same methodology as step5 but on the disease PPI network.
# ============================================================================

import pandas as pd
import networkx as nx
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR    = "/content/drive/MyDrive/Project"

NETWORK_FILE   = os.path.join(PROJECT_DIR, "results", "networks", "ppi_disease_BC.tsv")
PROXIMITY_FILE = os.path.join(PROJECT_DIR, "results", "genes",   "proximity_BC_disease_rwr.tsv")
OUTPUT_EDGES   = os.path.join(PROJECT_DIR, "results", "networks", "disease_subgraph_BC_rwr.tsv")

# ── 1. Load proximity results and filter significant genes ────────────────────
# Uses the 'significant' column from step10 (adj_pvalue_bh < 0.05).
print("Loading proximity results...")
prox_df = pd.read_csv(PROXIMITY_FILE, sep='\t')

sig_df    = prox_df[prox_df['significant'] == True].copy()
sig_genes = set(sig_df['gene_symbol'].tolist())

print(f"  Total genes tested                    : {len(prox_df)}")
print(f"  Significant (adj_pvalue_bh < 0.05)    : {len(sig_genes)}")

# ── 2. Load Disease PPI network ──────────────────────────────────────────────
print("\nLoading Disease PPI network...")
net_df = pd.read_csv(NETWORK_FILE, sep='\t')
print(f"  Total edges in Disease PPI  : {len(net_df)}")

# ── 3. Filter edges — BOTH endpoints must be in the significant gene set ─────
print("\nFiltering edges (both endpoints must be in significant gene set)...")
mask      = net_df['nodeA'].isin(sig_genes) & net_df['nodeB'].isin(sig_genes)
sub_edges = net_df[mask].copy()

print(f"  Edges after filtering   : {len(sub_edges)}")

# ── 4. Identify which genes actually appear in the subgraph ───────────────────
nodes_in_subgraph = set(sub_edges['nodeA']).union(set(sub_edges['nodeB']))
excluded = sig_genes - nodes_in_subgraph

print(f"  Genes in subgraph       : {len(nodes_in_subgraph)}")
print(f"  Significant genes with no internal edges (excluded): {len(excluded)}")
if excluded:
    print(f"    Excluded: {', '.join(sorted(excluded))}")

# ── 5. Save subgraph edge list ────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_EDGES), exist_ok=True)
sub_edges.to_csv(OUTPUT_EDGES, sep='\t', index=False)

# ── 6. Summary ───────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"SUBGRAPH EXTRACTION RESULTS (Disease PPI)")
print(f"{'='*60}")
print(f"  Input  : {len(sig_genes)} proximity-significant genes (BH FDR < 0.05)")
print(f"  Output : {len(nodes_in_subgraph)} genes, {len(sub_edges)} edges")
print(f"\n✅ Step 11 subgraph complete — saved to:\n   {OUTPUT_EDGES}")



Loading proximity results...
  Total genes tested                    : 9450
  Significant (adj_pvalue_bh < 0.05)    : 203

Loading Disease PPI network...
  Total edges in Disease PPI  : 87645

Filtering edges (both endpoints must be in significant gene set)...
  Edges after filtering   : 356
  Genes in subgraph       : 203
  Significant genes with no internal edges (excluded): 0

SUBGRAPH EXTRACTION RESULTS (Disease PPI)
  Input  : 203 proximity-significant genes (BH FDR < 0.05)
  Output : 203 genes, 356 edges

✅ Step 11 subgraph complete


## Step 12: Leiden Clustering (Disease PPI)Leiden clustering on the filtered disease PPI subgraph. Uses LCC and removes clusters with fewer than 20 genes.

In [ ]:
# ============================================================================
# step12_disease_clustering.py
# Leiden Clustering directly on Disease PPI Network
#
# Since the disease PPI (IntAct + BioGRID via PSICQUIC) is already
# disease-specific, we skip RWR/proximity/subgraph and cluster directly.
# Uses:
#   - Full disease PPI from step8
#   - LCC only
#   - Minimum cluster size of 20 genes
#
# References:
#   [1] Traag, V. A., Waltman, L., & van Eck, N. J. (2019). From Louvain
#       to Leiden: guaranteeing well-connected communities.
#       Scientific Reports, 9(1), 5233.
#       https://doi.org/10.1038/s41598-019-41695-z
#
#   [2] Traag, V. A. (2024). leidenalg: Leiden algorithm for community
#       detection. Python package.
#       https://github.com/vtraag/leidenalg
#
#   [3] Csárdi, G., Nepusz, T., et al. (2024). igraph: Network analysis
#       and visualization. R/Python package.
#       https://doi.org/10.5281/zenodo.7682609
#       https://github.com/igraph/python-igraph
# ============================================================================

import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR = "/content/drive/MyDrive/Project"

SUBGRAPH_FILE  = os.path.join(PROJECT_DIR, "results", "networks", "disease_subgraph_BC_rwr.tsv")
PROXIMITY_FILE = os.path.join(PROJECT_DIR, "results", "genes",    "proximity_BC_disease_rwr.tsv")
OUTPUT_FILE    = os.path.join(PROJECT_DIR, "results", "modules",  "modules_BC_psicquic.tsv")

MIN_CLUSTER_SIZE = 20

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# ── 1. Load disease subgraph + proximity data ───────────────────────────────
print("=" * 60)
print("STEP 12: Leiden Clustering on Disease PPI (filtered)")
print("=" * 60)

edges = pd.read_csv(SUBGRAPH_FILE, sep='\t')
print(f"\n  Disease subgraph edges: {len(edges)}")

# Load RWR scores for edge weights
prox_df = pd.read_csv(PROXIMITY_FILE, sep='\t')
rwr_lookup = dict(zip(prox_df["gene_symbol"], prox_df["rwr_score"]))

# Only use significant genes
sig_mask = prox_df["significant"] == True
selected_nodes = set(prox_df.loc[sig_mask, "gene_symbol"])

# ── 2. Build graph ───────────────────────────────────────────────────────────
G = nx.Graph()
for _, row in edges.iterrows():
    a, b = row["nodeA"], row["nodeB"]
    if a in selected_nodes and b in selected_nodes:
        rwr_a = rwr_lookup.get(a, 0.0)
        rwr_b = rwr_lookup.get(b, 0.0)
        rwr_edge_weight = (rwr_a + rwr_b) / 2.0
        G.add_edge(a, b, weight=rwr_edge_weight)

print(f"  Graph nodes: {G.number_of_nodes()}")
print(f"  Graph edges: {G.number_of_edges()}")

# ── 3. Keep Largest Connected Component ──────────────────────────────────────
lcc_nodes = max(nx.connected_components(G), key=len)
G = G.subgraph(lcc_nodes).copy()

print(f"  LCC nodes: {G.number_of_nodes()}")
print(f"  LCC edges: {G.number_of_edges()}")

# ── 4. Convert to igraph ─────────────────────────────────────────────────────
node_list       = list(G.nodes())
mapping         = {node: i for i, node in enumerate(node_list)}
reverse_mapping = {i: node for node, i in mapping.items()}

edges_with_weight = [
    (mapping[u], mapping[v], data["weight"])
    for u, v, data in G.edges(data=True)
]

ig_graph = ig.Graph(
    n        = len(node_list),
    edges    = [(u, v) for u, v, _ in edges_with_weight],
    directed = False
)
ig_graph.es["weight"] = [w for _, _, w in edges_with_weight]
ig_graph.vs["name"]   = node_list

# ── 5. Leiden clustering [1] ─────────────────────────────────────────────────
leiden_partition = leidenalg.find_partition(
    ig_graph,
    leidenalg.RBConfigurationVertexPartition,
    weights              = "weight",
    resolution_parameter = 0.8,
    seed                 = 42,
    n_iterations         = 50
)

print(f"\n  Clusters found: {len(leiden_partition)}")
print(f"  Modularity    : {leiden_partition.modularity:.4f}")

# ── 6. Extract results ───────────────────────────────────────────────────────
leiden_clusters = {
    reverse_mapping[node_id]: comm
    for node_id, comm in enumerate(leiden_partition.membership)
}

cluster_df = pd.DataFrame([
    {"gene": gene, "cluster": cluster}
    for gene, cluster in leiden_clusters.items()
])

# ── 7. Drop clusters < 20 genes ─────────────────────────────────────────────
valid_clusters = cluster_df["cluster"].value_counts()
valid_clusters = valid_clusters[valid_clusters >= MIN_CLUSTER_SIZE].index
cluster_df     = cluster_df[cluster_df["cluster"].isin(valid_clusters)]

cluster_df = cluster_df.sort_values(["cluster", "gene"]).reset_index(drop=True)

# ── 8. Summary ───────────────────────────────────────────────────────────────
print(f"\nCluster summary (clusters with >= {MIN_CLUSTER_SIZE} genes):")
if len(cluster_df) > 0:
    summary = cluster_df["cluster"].value_counts().sort_index()
    for c, n in summary.items():
        print(f"  Cluster {c}: {n} genes")
    print(f"\n  Total clusters: {len(summary)}")
    print(f"  Total genes:    {len(cluster_df)}")
else:
    print("  ⚠ No clusters with >= 20 genes")

# ── 9. Save ──────────────────────────────────────────────────────────────────
cluster_df.to_csv(OUTPUT_FILE, sep='\t', index=False)

print(f"\n✅ Step 12 complete — saved to:\n   {OUTPUT_FILE}")



STEP 12: Leiden Clustering on Disease PPI (filtered)

  Disease subgraph edges: 356
  Graph nodes: 203
  Graph edges: 356
  LCC nodes: 189
  LCC edges: 340

  Clusters found: 6
  Modularity    : 0.5826

Cluster summary (clusters with >= 20 genes):
  Cluster 0: 55 genes
  Cluster 1: 50 genes
  Cluster 2: 31 genes
  Cluster 3: 25 genes

  Total clusters: 4
  Total genes:    161

✅ Step 12 complete


## Step 13: Module Comparison (STRING vs Disease PPI)Compares the STRING and disease PPI modules using ARI, NMI, and Jaccard similarity. Also performs granularity matching by merging disease modules that map to the same STRING module.

In [ ]:
# ============================================================================
# step13_compare.py
# Module Comparison: STRING vs Disease PPI Clusters
#
# Compares Leiden clustering from the STRING pipeline (step6) with the
# disease PPI pipeline (step11) using ARI, NMI, and Jaccard similarity.
#
# References:
#   [1] Hubert, L. & Arabie, P. (1985). Comparing partitions.
#       Journal of Classification, 2(1), 193–218.
#   [2] Vinh, N. X. et al. (2010). Information theoretic measures for
#       clusterings comparison. JMLR, 11, 2837–2854.
# ============================================================================

import pandas as pd
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from itertools import product, combinations
import os

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR = "/content/drive/MyDrive/Project"

STRING_MODULES  = os.path.join(PROJECT_DIR, "results", "modules", "modules_BC_leiden.tsv")
DISEASE_MODULES = os.path.join(PROJECT_DIR, "results", "modules", "modules_BC_psicquic.tsv")
REPORT_DIR      = os.path.join(PROJECT_DIR, "results", "reports")

os.makedirs(REPORT_DIR, exist_ok=True)

print("=" * 60)
print("STEP 13: Module Comparison (STRING vs Disease PPI)")
print("=" * 60)

# ── 1. Load module assignments ───────────────────────────────────────────────
df_string = pd.read_csv(STRING_MODULES, sep="\t")
if "gene_symbol" in df_string.columns:
    df_string = df_string.rename(columns={"gene_symbol": "gene"})
if "cluster" in df_string.columns and "module_id" not in df_string.columns:
    df_string = df_string.rename(columns={"cluster": "module_id"})

df_disease = pd.read_csv(DISEASE_MODULES, sep="\t")
if "gene_symbol" in df_disease.columns:
    df_disease = df_disease.rename(columns={"gene_symbol": "gene"})
if "cluster" in df_disease.columns and "module_id" not in df_disease.columns:
    df_disease = df_disease.rename(columns={"cluster": "module_id"})

print(f"\nSTRING modules  : {df_string['module_id'].nunique()} modules, "
      f"{df_string['gene'].nunique()} unique genes")
print(f"Disease modules : {df_disease['module_id'].nunique()} modules, "
      f"{df_disease['gene'].nunique()} unique genes")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. Jaccard module-pair similarity table
# ═══════════════════════════════════════════════════════════════════════════════
string_modules  = df_string.groupby("module_id")["gene"].apply(set).to_dict()
disease_modules = df_disease.groupby("module_id")["gene"].apply(set).to_dict()

jaccard_rows = []
for (s_mod, s_genes), (d_mod, d_genes) in product(
        string_modules.items(), disease_modules.items()):
    intersection = s_genes & d_genes
    union        = s_genes | d_genes
    jaccard = round(len(intersection) / len(union), 4) if union else 0.0
    jaccard_rows.append({
        "disease":         "BC",
        "string_module":   s_mod,
        "string_size":     len(s_genes),
        "disease_module":  d_mod,
        "disease_size":    len(d_genes),
        "overlap_size":    len(intersection),
        "jaccard":         jaccard,
        "overlap_genes":   ";".join(sorted(intersection)) if intersection else ""
    })

jaccard_df = pd.DataFrame(jaccard_rows).sort_values(
    ["string_module", "jaccard"], ascending=[True, False]
)
jaccard_df.to_csv(
    os.path.join(REPORT_DIR, "module_pair_similarity.tsv"),
    sep="\t", index=False
)

# Best match for each disease module
best_string_for_disease = (
    jaccard_df[jaccard_df["overlap_size"] > 0]
    .sort_values("jaccard", ascending=False)
    .drop_duplicates(subset="disease_module")
    .set_index("disease_module")["string_module"]
    .to_dict()
)

print(f"\nDisease PPI → STRING module mapping (by best Jaccard):")
for d_mod, s_mod in sorted(best_string_for_disease.items(),
                            key=lambda x: str(x[0])):
    print(f"  Disease {d_mod}  →  STRING {s_mod}")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. Merge disease modules that map to the same STRING module
# ═══════════════════════════════════════════════════════════════════════════════
df_disease_merged = df_disease.copy()
df_disease_merged["module_id_merged"] = df_disease_merged["module_id"].map(
    lambda m: best_string_for_disease.get(m, m)
)

print(f"\nAfter merging disease modules to match STRING granularity:")
for merged_id, grp in df_disease_merged.groupby("module_id_merged"):
    orig_ids = sorted(grp["module_id"].unique())
    print(f"  Merged {orig_ids}  →  '{merged_id}'  ({len(grp)} genes)")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. Compute ARI [1], NMI [2], Jaccard
# ═══════════════════════════════════════════════════════════════════════════════
merged_raw = pd.merge(
    df_string[["gene", "module_id"]],
    df_disease[["gene", "module_id"]],
    on="gene", suffixes=("_string", "_disease")
)

merged_fixed = pd.merge(
    df_string[["gene", "module_id"]],
    df_disease_merged[["gene", "module_id_merged"]],
    on="gene"
)

print(f"\nShared universe (raw)   : {len(merged_raw)} genes")
print(f"Shared universe (merged): {len(merged_fixed)} genes")

report_rows = []

# Jaccard clustering similarity
def jaccard_clustering(labels1, labels2):
    n = len(labels1)
    both = 0
    either = 0
    for i, j in combinations(range(n), 2):
        same_1 = labels1[i] == labels1[j]
        same_2 = labels2[i] == labels2[j]
        if same_1 and same_2:
            both += 1
        if same_1 or same_2:
            either += 1
    return both / either if either > 0 else 0.0

if len(merged_raw) >= 2:
    nmi_raw = normalized_mutual_info_score(
        merged_raw["module_id_string"], merged_raw["module_id_disease"])
    ari_raw = adjusted_rand_score(
        merged_raw["module_id_string"], merged_raw["module_id_disease"])
    jac_raw = jaccard_clustering(
        merged_raw["module_id_string"].tolist(),
        merged_raw["module_id_disease"].tolist()
    )
    report_rows += [
        {"disease": "BC", "viewA": "string_leiden", "viewB": "disease_leiden",
         "comparison": "raw", "metric": "NMI",
         "universe_size": len(merged_raw), "value": round(nmi_raw, 4)},
        {"disease": "BC", "viewA": "string_leiden", "viewB": "disease_leiden",
         "comparison": "raw", "metric": "ARI",
         "universe_size": len(merged_raw), "value": round(ari_raw, 4)},
        {"disease": "BC", "viewA": "string_leiden", "viewB": "disease_leiden",
         "comparison": "raw", "metric": "Jaccard",
         "universe_size": len(merged_raw), "value": round(jac_raw, 4)},
    ]

if len(merged_fixed) >= 2:
    nmi_merged = normalized_mutual_info_score(
        merged_fixed["module_id"], merged_fixed["module_id_merged"])
    ari_merged = adjusted_rand_score(
        merged_fixed["module_id"], merged_fixed["module_id_merged"])
    report_rows += [
        {"disease": "BC", "viewA": "string_leiden", "viewB": "disease_merged",
         "comparison": "granularity_matched", "metric": "NMI",
         "universe_size": len(merged_fixed), "value": round(nmi_merged, 4)},
        {"disease": "BC", "viewA": "string_leiden", "viewB": "disease_merged",
         "comparison": "granularity_matched", "metric": "ARI",
         "universe_size": len(merged_fixed), "value": round(ari_merged, 4)},
    ]

report_df = pd.DataFrame(report_rows)
report_df.to_csv(
    os.path.join(REPORT_DIR, "module_comparison.tsv"),
    sep="\t", index=False
)

# ── 5. Print summary ─────────────────────────────────────────────────────────
print(f"\n{'─' * 50}")
print("Comparison Metrics:")
print(f"{'─' * 50}")
if not report_df.empty:
    print(report_df.to_string(index=False))
else:
    print("  No overlapping genes found between STRING and Disease PPI modules.")

if report_rows:
    ari_r = next((r["value"] for r in report_rows
                  if r["metric"] == "ARI" and r["comparison"] == "raw"), None)
    ari_m = next((r["value"] for r in report_rows
                  if r["metric"] == "ARI"
                  and r["comparison"] == "granularity_matched"), None)
    if ari_r is not None and ari_m is not None:
        print(f"\nARI improvement from granularity matching: "
              f"{ari_r:.4f} → {ari_m:.4f} (+{ari_m - ari_r:.4f})")

# ── 6. Best match summary ────────────────────────────────────────────────────
print(f"\n{'─' * 50}")
print("Best disease PPI match for each STRING module:")
print(f"{'─' * 50}")
best_matches = (
    jaccard_df[jaccard_df["overlap_size"] > 0]
    .sort_values("jaccard", ascending=False)
    .drop_duplicates(subset="string_module")
    .sort_values("string_module")
)
for _, row in best_matches.iterrows():
    print(f"  STRING {row['string_module']} (n={row['string_size']}) "
          f"<-> Disease {row['disease_module']} (n={row['disease_size']}) "
          f"| overlap={row['overlap_size']}  Jaccard={row['jaccard']}")
    if row["overlap_genes"]:
        print(f"    shared: {row['overlap_genes']}")

print(f"\n{'─' * 50}")
print(f"Saved: {os.path.join(REPORT_DIR, 'module_comparison.tsv')}")
print(f"Saved: {os.path.join(REPORT_DIR, 'module_pair_similarity.tsv')}")
print(f"\n✅ Step 13 complete — Module comparison done")



STEP 13: Module Comparison (STRING vs Disease PPI)

STRING modules  : 4 modules, 237 unique genes
Disease modules : 4 modules, 161 unique genes

Comparison Metrics:
disease         viewA          viewB          comparison  metric  universe_size  value
     BC string_leiden disease_leiden                 raw     NMI             32 0.5547
     BC string_leiden disease_leiden                 raw     ARI             32 0.3718
     BC string_leiden disease_leiden                 raw Jaccard             32 0.3839

Best disease PPI match for each STRING module:
  STRING 0 (n=94) <-> Disease 2 (n=31) | overlap=8
    shared: AKT1;CAB39;CAB39L;CASP8;PTEN;STK11;STRADA;STRADB
  STRING 1 (n=84) <-> Disease 1 (n=50) | overlap=8
    shared: BRCA2;FIGNL1;PALB2;RAD51;RAD51C;SPIDR;XRCC2;XRCC3
  STRING 2 (n=34) <-> Disease 0 (n=55) | overlap=1
    shared: TP53
  STRING 3 (n=25) <-> Disease 0 (n=55) | overlap=4
    shared: ATM;CBY2;MLH1;NABP2

✅ Step 13 complete — Module comparison done


## Step 14: Pathway Enrichment (Disease PPI Modules)Runs KEGG and Reactome enrichment for each disease PPI module.

In [ ]:
# ============================================================================
# step14_enrichment.py
# Pathway Enrichment for Disease PPI Modules
#
# Runs KEGG and Reactome enrichment on each disease PPI Leiden module
# from step12. Produces per-module CSV files and bar plots, plus a
# combined enrichment summary across all modules.
#
# Notes:
#   1. Skips plotting/saving when a cluster has no enrichment results
#   2. Uses only Adjusted P-value < 0.05 for significance (removes
#      non-significant terms before saving/plotting)
#   3. Saves to per-module folders (no overwriting)
#
# References:
#   [1] Kanehisa, M. & Goto, S. (2000). KEGG: Kyoto Encyclopedia of Genes
#       and Genomes. Nucleic Acids Research, 28(1), 27–30.
#       https://doi.org/10.1093/nar/28.1.27
#
#   [2] Jassal, B. et al. (2020). The Reactome pathway knowledgebase.
#       Nucleic Acids Research, 48(D1), D498–D503.
#       https://doi.org/10.1093/nar/gkz1031
# ============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import gseapy as gp
except ImportError:
    raise ImportError("gseapy is not installed. Run: pip install gseapy")

# ── Configuration ─────────────────────────────────────────────────────────────
PROJECT_DIR = "/content/drive/MyDrive/Project"

MODULES_FILE   = os.path.join(PROJECT_DIR, "results", "modules", "modules_BC_psicquic.tsv")
ENRICHMENT_DIR = os.path.join(PROJECT_DIR, "results", "enrichment_disease")

GENE_SETS = ["KEGG_2021_Human", "Reactome_2022"]
P_CUTOFF  = 0.05

os.makedirs(ENRICHMENT_DIR, exist_ok=True)

print("=" * 60)
print("STEP 14: Pathway Enrichment for Disease PPI Modules")
print("=" * 60)

# ── 1. Load module assignments ───────────────────────────────────────────────
modules_df = pd.read_csv(MODULES_FILE, sep="\t")
if "module_id" not in modules_df.columns and "cluster" in modules_df.columns:
    modules_df["module_id"] = modules_df["cluster"]
module_ids = sorted(modules_df["module_id"].unique())

print(f"\nLoaded {len(modules_df)} genes across {len(module_ids)} modules")

# ── 2. Per-module enrichment ─────────────────────────────────────────────────
all_results = []

for mod_id in module_ids:
    mod_genes = modules_df[modules_df["module_id"] == mod_id]["gene"].tolist()

    print(f"\n{'─' * 50}")
    print(f"Module {mod_id}: {len(mod_genes)} genes")
    print(f"  Genes: {', '.join(sorted(mod_genes)[:15])}{'...' if len(mod_genes) > 15 else ''}")

    if len(mod_genes) < 3:
        print("  ⚠ Too few genes for enrichment — skipping")
        continue

    # ── Run Enrichr (no cutoff here — filter manually after) ─────────────────
    mod_dir = os.path.join(ENRICHMENT_DIR, f"module_{mod_id}")
    os.makedirs(mod_dir, exist_ok=True)

    try:
        enr = gp.enrichr(
            gene_list=mod_genes,
            gene_sets=GENE_SETS,
            organism="human",
            outdir=os.path.join(mod_dir, "enrichr_raw"),
            cutoff=1.0,        # get ALL terms, filter below
            no_plot=True,
        )
        enrich_df = enr.results
    except Exception as e:
        print(f"  ❌ Enrichr failed: {e}")
        continue

    # 1. Skip empty results
    if enrich_df is None or enrich_df.empty:
        print(f"  No enrichment results → skip")
        continue

    # 2. Use Adjusted P-value only — remove non-significant terms
    enrich_df = enrich_df[enrich_df["Adjusted P-value"] < P_CUTOFF].copy()

    if enrich_df.empty:
        print(f"  No significant pathways (Adjusted P-value < {P_CUTOFF}) → skip")
        continue

    # Add module metadata
    enrich_df["module_id"]   = mod_id
    enrich_df["module_size"] = len(mod_genes)
    all_results.append(enrich_df)

    # 3. Save per-module (no overwriting — each module has its own folder)
    enrich_df.to_csv(os.path.join(mod_dir, "enrichment_full.csv"), index=False)

    # ── Per-database CSV and plots ──────────────────────────────────────────
    for db_name in GENE_SETS:
        db_df = (enrich_df[enrich_df["Gene_set"] == db_name]
                 .sort_values("Adjusted P-value")
                 .head(10)
                 .copy())

        if db_df.empty:
            continue

        short_name = db_name.split("_")[0].lower()
        db_df.to_csv(os.path.join(mod_dir, f"top_{short_name}.csv"), index=False)

        # Plot
        db_df["-log10(padj)"] = -np.log10(
            db_df["Adjusted P-value"].replace(0, 1e-300))
        db_df["Term"] = db_df["Term"].apply(
            lambda x: x if len(str(x)) <= 55 else str(x)[:52] + "...")

        fig, ax = plt.subplots(figsize=(10, max(3, len(db_df) * 0.4)))
        color = "skyblue" if "KEGG" in db_name else "lightgreen"
        sns.barplot(data=db_df, y="Term", x="-log10(padj)", color=color, ax=ax)
        ax.set_title(f"Module {mod_id} — Top {db_name} Pathways")
        ax.set_xlabel("-log10(Adjusted P-value)")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(mod_dir, f"top_{short_name}.png"), dpi=150)
        plt.close()

        # Print top 5
        print(f"\n  Top 5 {db_name}:")
        for _, row in db_df.head(5).iterrows():
            print(f"    {row['Term']:<50} padj={row['Adjusted P-value']:.2e}")

# ── 3. Combined enrichment results ───────────────────────────────────────────
if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(
        os.path.join(ENRICHMENT_DIR, "all_modules_enrichment.csv"),
        index=False
    )
    print(f"\n{'=' * 60}")
    print(f"ENRICHMENT SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Modules with enrichment : {combined['module_id'].nunique()}")
    print(f"  Total significant terms : {len(combined)}")
    print(f"  Combined CSV saved      : {os.path.join(ENRICHMENT_DIR, 'all_modules_enrichment.csv')}")
else:
    print("\n⚠ No enrichment results found for any module")

print(f"\n✅ Step 14 complete — Disease PPI enrichment done")
print(f"   Results in: {ENRICHMENT_DIR}")



---## Summary| Metric | STRING Pipeline | Disease PPI Pipeline ||--------|----------------|---------------------|| Network source | STRING v12.0 | IntAct PSICQUIC + BioGRID || Total edges | 236,333 | 87,645 || After proximity | 253 significant genes | 203 significant genes || Subgraph | 252 genes, ~2,400 edges | 203 genes, 356 edges || Clusters | 4 (modularity 0.34) | 4 (modularity 0.58) || **ARI** | — | **0.37** || **NMI** | — | **0.55** |The moderate ARI (0.37) and strong NMI (0.55) indicate that the core modular structure is preserved across independent network sources, validating the biological relevance of the identified breast cancer modules.